<a href="https://colab.research.google.com/github/mts924/Backtest-Python-NVIDIA/blob/main/backtest_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance backtesting pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.8/190.8 kB 2.4 MB/s eta 0:00:00


In [ ]:
import yfinance as yf
import pandas as pd
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

# 1. Download NVDA historical data
print("Fetching Nvidia data...")
nvda = yf.Ticker("NVDA")
data = nvda.history(period="5y")

# Keep only the columns backtesting.py needs
data = data[['Open', 'High', 'Low', 'Close', 'Volume']]

# Helper function to calculate Simple Moving Average
def SMA(values, n):
    return pd.Series(values).rolling(n).mean()

# 2. Define the Trading Strategy
class SmaCross(Strategy):
    n1 = 20  # Fast moving average (20 days)
    n2 = 50  # Slow moving average (50 days)

    def init(self):
        # Pre-calculate indicators
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)

    def next(self):
        # If the 20-day crosses ABOVE the 50-day -> BUY
        if crossover(self.sma1, self.sma2):
            self.buy()
        # If the 50-day crosses ABOVE the 20-day -> SELL
        elif crossover(self.sma2, self.sma1):
            self.position.close()

# 3. Run the Simulation
print("Running backtest...")
# Starting with $10,000 and 0.2% broker commission per trade
bt = Backtest(data, SmaCross, cash=10000, commission=0.002)
stats = bt.run()

print("\n--- PERFORMANCE METRICS ---")
print(stats)

# Generate an interactive chart
bt.plot()

Fetching Nvidia data...
Running backtest...


Backtest.run:   0%|          | 0/1205 [00:00<?, ?bar/s]

/tmp/ipykernel_2431/884322828.py:40: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()



--- PERFORMANCE METRICS ---
Start                     2021-09-29 00:00...
End                       2026-09-29 00:00...
Duration                   1826 days 00:00:00
Exposure Time [%]                    55.05976
Equity Final [$]                  27391.40575
Equity Peak [$]                   31452.91087
Commissions [$]                      924.0043
Return [%]                          173.91406
Buy & Hold Return [%]                619.1946
Return (Ann.) [%]                    22.44516
Volatility (Ann.) [%]                46.38948
CAGR [%]                             22.33053
Sharpe Ratio                          0.48384
Sortino Ratio                         0.97337
Calmar Ratio                          0.38792
Alpha [%]                          -123.64753
Beta                                  0.48056
Max. Drawdown [%]                   -57.85991
Avg. Drawdown [%]                    -7.90248
Max. Drawdown Duration      442 days 00:00:00
Avg. Drawdown Duration       58 days 00:00:00
# Tra

/usr/local/lib/python3.13/dist-packages/bokeh/util/serialization.py:242: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1356', ...)

In [5]:
import yfinance as yf
import pandas as pd
import warnings
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

# Désactiver les alertes pour empêcher Colab de geler
warnings.filterwarnings("ignore")

# 1. Fetch Data
nvda = yf.Ticker("NVDA")
data = nvda.history(period="5y")
data = data[['Open', 'High', 'Low', 'Close', 'Volume']]

def SMA(values, n):
    return pd.Series(values).rolling(n).mean()

# 2. Define Strategy with Stop-Loss
class SmaCross(Strategy):
    n1 = 20
    n2 = 50

    def init(self):
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)

    def next(self):
        if crossover(self.sma1, self.sma2):
            sl_price = self.data.Close[-1] * 0.95
            self.buy(sl=sl_price)

        elif crossover(self.sma2, self.sma1):
            self.position.close()

bt = Backtest(data, SmaCross, cash=10000, commission=0.002)

# 4. Run Optimization (sans max_workers !)
print("Running optimization...")
stats_opt = bt.optimize(
    n1=range(5, 30, 5),
    n2=range(10, 70, 5),
    maximize='Sharpe Ratio',
    constraint=lambda param: param.n1 < param.n2
)

# 5. Show Results
print(stats_opt)
print("\nBest Parameters Found:", stats_opt._strategy)
bt.plot()

Running optimization...


Backtest.optimize:   0%|          | 0/50 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/1219 [00:00<?, ?bar/s]

Start                     2021-10-01 00:00...
End                       2026-09-30 00:00...
Duration                   1825 days 00:00:00
Exposure Time [%]                    52.95056
Equity Final [$]                  55921.53449
Equity Peak [$]                   72915.49935
Commissions [$]                    3533.88467
Return [%]                          459.21534
Buy & Hold Return [%]               624.18607
Return (Ann.) [%]                      41.368
Volatility (Ann.) [%]                49.45377
CAGR [%]                             41.12964
Sharpe Ratio                           0.8365
Sortino Ratio                         2.08416
Calmar Ratio                          1.18351
Alpha [%]                           195.37581
Beta                                  0.42269
Max. Drawdown [%]                   -34.95371
Avg. Drawdown [%]                    -6.67116
Max. Drawdown Duration      263 days 00:00:00
Avg. Drawdown Duration       35 days 00:00:00
# Trades                          

GridPlot(id='p1735', ...)